# MMNet Demo

This notebook demonstrates how to run the Motor-unit Mode Network (MMNet) implementation
(`mmnet.py`) on motor unit discharge-rate data and inspect the
resulting latent motor unit modes.

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf

from mmnet_vaelstm import (
    Config,
    set_global_determinism,
    load_mat_concatenated_data,
    zscore_per_mu,
    trim_edges,
    make_overlapping_windows,
    train_val_split,
    make_tf_datasets,
    SequenceLatentVAE,
    fit,
    reconstruct_full_signal,
    extract_latent_full,
    variance_explained_percent,
)


In [ ]:
# User settings
mat_path = "All_MUs_S01s35bef.mat"   # change to your file
mat_key = "concatenated_data"

cfg = Config(
    seed=42,
    fs_hz=2000,
    trim_s=0.4,
    window=1000,
    step=250,
    latent_dim=4,
    epochs=200,
    batch_size=8,
    lr=1e-4,
    beta_kl=0.1,
)

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True, parents=True)
ckpt_path = out_dir / f"best_weights_latent{cfg.latent_dim}.weights.h5"

set_global_determinism(cfg.seed)


In [ ]:
# Load and preprocess
data_MxT = load_mat_concatenated_data(mat_path, key=mat_key)
data_MxT = zscore_per_mu(data_MxT, eps=cfg.eps)
data_TxM = trim_edges(data_MxT.T, fs_hz=cfg.fs_hz, trim_s=cfg.trim_s)

windows = make_overlapping_windows(data_TxM, window=cfg.window, step=cfg.step)
train_x, val_x = train_val_split(windows, val_frac=cfg.val_frac, seed=cfg.seed)
train_ds, val_ds = make_tf_datasets(train_x, val_x, batch_size=cfg.batch_size)

print('Signal shape after trimming:', data_TxM.shape)
print('Windowed dataset shape:', windows.shape)


In [ ]:
# Build and train model
n_mus = data_TxM.shape[1]
model = SequenceLatentVAE(
    n_mus=n_mus,
    window=cfg.window,
    latent_dim=cfg.latent_dim,
    dropout=cfg.dropout,
)

_ = model(tf.zeros((1, cfg.window, n_mus), dtype=tf.float32), training=False)
history = fit(model, train_ds, val_ds, cfg, mu_groups=None, ckpt_path=ckpt_path)

if ckpt_path.exists():
    model.load_weights(str(ckpt_path))

np.save(out_dir / 'history.npy', history, allow_pickle=True)


In [ ]:
# Optional full-signal reconstruction
recon_full = reconstruct_full_signal(model, data_TxM, window=cfg.window, step=cfg.step)
np.save(out_dir / 'recon_full.npy', recon_full)

ve = variance_explained_percent(data_TxM, recon_full, eps=cfg.eps)
print(f'Full-signal variance explained: {ve:.2f}%')


In [ ]:
# Optional full-length latent time series
latent_full = extract_latent_full(model, data_TxM, window=cfg.window, step=cfg.step)
np.save(out_dir / 'latent_full.npy', latent_full)

print('Saved outputs to:', out_dir.resolve())
print('latent_full shape:', latent_full.shape)
